<a href="https://colab.research.google.com/github/Luca4Spreafico/CHALLENGE-2---Ibuprofen/blob/main/modify_the_dataset_by_Luca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🌐 **Google Drive setup**

In [2]:
from google.colab import drive

import os
import shutil
import numpy as np
from PIL import Image
import pandas as pd
from pathlib import Path

# Define your working directory
drive.mount("/gdrive")
working_dir = "/gdrive/My Drive/B University/Artificial Networks/an2dl2526c2"
%cd $working_dir

# Define paths
dataset_dir = working_dir

train_data_dir = os.path.join(dataset_dir, "train_data_clean")
train_labels_path = os.path.join(dataset_dir, "train_labels_clean.csv")
test_data_dir = os.path.join(dataset_dir, "test_data")

def create_organized_folders(data_dir, output_base_dir, dataset_type="train"):
    """
    Organize images and masks into separate folders and create masked images.

    Args:
        data_dir: Source directory containing mixed images and masks
        output_base_dir: Base directory for organized output
        dataset_type: "train" or "test" to distinguish datasets
    """

    # Create output directories
    images_dir = os.path.join(output_base_dir, f"{dataset_type}_images")
    masks_dir = os.path.join(output_base_dir, f"{dataset_type}_masks")
    masked_images_dir = os.path.join(output_base_dir, f"{dataset_type}_masked_images")

    os.makedirs(images_dir, exist_ok=True)
    os.makedirs(masks_dir, exist_ok=True)
    os.makedirs(masked_images_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Processing {dataset_type} dataset...")
    print(f"{'='*60}")

    # Get all files in the directory
    all_files = sorted([f for f in os.listdir(data_dir) if f.endswith('.png')])

    # Separate images and masks
    image_files = [f for f in all_files if f.startswith('img_')]
    mask_files = [f for f in all_files if f.startswith('mask_')]
    #
    print(f"Found {len(image_files)} images")
    print(f"Found {len(mask_files)} masks")

    # Verify that each image has a corresponding mask
    if len(image_files) != len(mask_files):
        print(f"WARNING: Number of images ({len(image_files)}) doesn't match masks ({len(mask_files)})")
    #
    # Process each image-mask pair
    processed_count = 0
    for img_file in image_files:
        # Extract the number from filename (e.g., img_0000.png -> 0000)
        img_number = img_file.split('_')[1].split('.')[0]
        mask_file = f"mask_{img_number}.png"

        # Check if corresponding mask exists
        if mask_file not in mask_files:
            print(f"WARNING: No mask found for {img_file}")
            continue

        # Full paths
        img_path = os.path.join(data_dir, img_file)
        mask_path = os.path.join(data_dir, mask_file)

        # Copy image to images folder
        shutil.copy2(img_path, os.path.join(images_dir, img_file))

        # Copy mask to masks folder
        shutil.copy2(mask_path, os.path.join(masks_dir, mask_file))

        # Create masked image (image * mask)
        try:
            # Load image and mask
            img = Image.open(img_path).convert('RGB')
            mask = Image.open(mask_path).convert('L')  # Convert mask to grayscale

            # Convert to numpy arrays
            img_array = np.array(img, dtype=np.float32)
            mask_array = np.array(mask, dtype=np.float32)
            #
            # Normalize mask to [0, 1] range (binary mask)
            mask_array = mask_array / 255.0
            #
            # Expand mask dimensions to match image channels (H, W) -> (H, W, 3)
            mask_array = np.expand_dims(mask_array, axis=2)
            mask_array = np.repeat(mask_array, 3, axis=2)

             # Apply mask:
            # - Where mask = 1: keep original image pixels (including blacks)
            # - Where mask = 0: set to white (255)
            masked_img_array = np.where(
                mask_array == 1,  # Condition: where mask is 1 (foreground)
                img_array,        # True: keep original image
                255               # False: set to white
            )

            # Convert back to uint8 and save
            masked_img = Image.fromarray(masked_img_array.astype(np.uint8))
            masked_img.save(os.path.join(masked_images_dir, f"masked_{img_number}.png"))

            processed_count += 1

            if processed_count % 100 == 0:
                print(f"Processed {processed_count}/{len(image_files)} image-mask pairs...")

        except Exception as e:
            print(f"ERROR processing {img_file}: {str(e)}")
            continue

    print(f"\n✓ Successfully processed {processed_count} image-mask pairs")
    print(f"✓ Images saved to: {images_dir}")
    print(f"✓ Masks saved to: {masks_dir}")
    print(f"✓ Masked images saved to: {masked_images_dir}")

    return images_dir, masks_dir, masked_images_dir


def load_labels(labels_path):
    """Load and display label distribution."""
    if not os.path.exists(labels_path):
        print(f"Labels file not found at {labels_path}")
        return None

    df = pd.read_csv(labels_path)
    print(f"\nLabel distribution:")
    print(df.iloc[:, 1].value_counts())

    return df


# Main execution
if __name__ == "__main__":
    print("Starting image and mask organization...")

    # Create output directory
    output_dir = os.path.join(working_dir, "organized_data")
    os.makedirs(output_dir, exist_ok=True)

    # Process training data
    if os.path.exists(train_data_dir):
        train_imgs, train_masks, train_masked = create_organized_folders(
            train_data_dir, output_dir, "train"
        )

        # Load and display training labels
        train_labels = load_labels(train_labels_path)
    else:
        print(f"Training directory not found: {train_data_dir}")

    # Process test data

    if os.path.exists(test_data_dir):
        test_imgs, test_masks, test_masked = create_organized_folders(
            test_data_dir, output_dir, "test"
        )
    else:
        print(f"Test directory not found: {test_data_dir}")


    print("\n" + "="*60)
    print("✓ All done! Your data is now organized.")
    print("="*60)

Mounted at /gdrive
/gdrive/My Drive/B University/Artificial Networks/an2dl2526c2
Starting image and mask organization...

Processing train dataset...
Found 581 images
Found 581 masks
Processed 100/581 image-mask pairs...
Processed 200/581 image-mask pairs...
Processed 300/581 image-mask pairs...
Processed 400/581 image-mask pairs...
Processed 500/581 image-mask pairs...

✓ Successfully processed 581 image-mask pairs
✓ Images saved to: /gdrive/My Drive/B University/Artificial Networks/an2dl2526c2/organized_data/train_images
✓ Masks saved to: /gdrive/My Drive/B University/Artificial Networks/an2dl2526c2/organized_data/train_masks
✓ Masked images saved to: /gdrive/My Drive/B University/Artificial Networks/an2dl2526c2/organized_data/train_masked_images

Label distribution:
label
Luminal B          204
Luminal A          158
HER2(+)            150
Triple negative     69
Name: count, dtype: int64

Processing test dataset...
Found 477 images
Found 477 masks
Processed 100/477 image-mask pairs.